# Baseline Comparison

Runs all baseline models on the same evaluation sets and produces a unified comparison table.

| Model | Type | Notes |
|---|---|---|
| Presidio | Rule-based | Microsoft open-source PHI anonymizer |
| GLiNER | Zero-shot NER | `urchade/gliner_mediumv2.1` |
| RoBERTa-i2b2 | Fine-tuned NER | `obi/deid_roberta_i2b2` |
| scispaCy | Medical NER | `en_core_sci_lg` (not de-id specific) |
| EdgeDeID-SFT | Ours | Qwen3-0.6B SFT |
| EdgeDeID-GRPO | Ours | Qwen3-0.6B SFT + GRPO |

## 0 — Setup

In [1]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
from tqdm import tqdm

from pipeline.data.preprocessing import preprocess_text
from pipeline.data.xml_utils import convert_to_xml
from pipeline.eval import compute_metrics, print_report, to_dataframe

# ── dataset selection ────────────────────────────────────────────────────────
# "synth" = held-out synthetic test set (LLM Eval.xlsx)
# "ext"   = external English benchmark
# "test"  = test.csv from HF dataset (data/test.csv)
DATASET = "test"   # change to "synth" or "ext" as needed

if DATASET == "synth":
    raw_df = pd.read_excel("../synth_eval/LLM Eval.xlsx")
    raw_df.rename(columns={"Revision": "default_response_1"}, inplace=True)
    raw_df = preprocess_text(raw_df, input_col="default_response_0", output_col="default_response_1")
    raw_df["GT-XML"] = raw_df["default_response_1"].apply(convert_to_xml)
    NOTE_COL = "default_response_0"
elif DATASET == "ext":
    raw_df = pd.read_csv("../ext_eval/en_data.csv")
    raw_df = preprocess_text(raw_df, input_col="Report", output_col="DeID")
    raw_df["GT-XML"] = raw_df["DeID"].apply(convert_to_xml)
    NOTE_COL = "Report"
elif DATASET == "test":
    raw_df = pd.read_csv("../../data/test.csv")
    print(raw_df.shape)
    raw_df = preprocess_text(raw_df, input_col="original", output_col="anonymized")
    raw_df["GT-XML"] = raw_df["anonymized"].apply(convert_to_xml)
    NOTE_COL = "original"
    print(raw_df.shape)
else:
    raise ValueError(f"Unknown DATASET: {DATASET!r}. Choose 'synth', 'ext', or 'test'.")

for tag in ["<anonymized_report>", "</anonymized_report>", "<anonymized_report/>",
            "<raw_report>", "</raw_report>", "<raw_report/>"]:
    raw_df[NOTE_COL] = raw_df[NOTE_COL].str.replace(tag, "", regex=False).str.strip()

notes     = raw_df[NOTE_COL].tolist()
gt_xmls   = raw_df["GT-XML"].tolist()
print(f"{len(notes)} evaluation samples | dataset={DATASET}")

(404, 3)
Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name              700            -
  age               332            -
  date              589            -
  place               0            -
  address           214            -
  phone             221            -
  email             215            -
  id                232            -
  url               293            -
  device            228            -

Text processing pipeline completed!
(377, 4)
377 evaluation samples | dataset=test


In [10]:
raw_df.head()

,anonymized,original,examples,GT-XML
0,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","(2024), Liam Novak (URL: https://myhealthonlin...",NONE,"<phi_entities>\n<names>\n<name id=""1"" type=""pa..."
1,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","On 2023, Liam Novak, a 68y male patient with a...",NONE,"<phi_entities>\n<names>\n<name id=""1"" type=""pa..."
2,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Dear Dr. Larsson, I hope this email finds you ...",NONE,"<phi_entities>\n<names>\n<name id=""1"" type=""pr..."
3,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Mr. Pt, email: d.kim1980@email.com, visited Me...",NONE,"<phi_entities>\n<names>\n<name id=""1"" type=""pr..."
4,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, a 58year-old man residing at...",NONE,"<phi_entities>\n<names>\n<name id=""1"" type=""pa..."


In [2]:
from pipeline.data.preprocessing import preprocess_text, inspect_changes
raw_df = pd.read_csv("../../data/test.csv")
before_col = raw_df['anonymized'].copy()          # snapshot before
df = preprocess_text(raw_df, input_col='original', output_col='anonymized')

# All changed rows (default 5 samples)
inspect_changes(before_col, df['anonymized'])

# Filter to a specific entity type
inspect_changes(before_col, df['anonymized'], entity='date', n=3)
inspect_changes(before_col, df['anonymized'], entity='phone')

# More/less context lines around each change
inspect_changes(before_col, df['anonymized'], entity='name', context=1)


Starting text processing pipeline...
Fixing spaces in closing tags...
Fixing spaces in tags...
Processing phone numbers...
Processing dates...
Processing ages...
Correcting gender terms...
Processing name tags...
Simplifying name tags...
Reformatting ID tags...
Processing remaining dates...
Removing markdown tags...

Processing Summary:
  Entity       Detected  Reformatted
  ----------  ---------  -----------
  name              700            -
  age               332            -
  date              589            -
  place               0            -
  address           214            -
  phone             221            -
  email             215            -
  id                232            -
  url               293            -
  device            228            -

Text processing pipeline completed!
376 row(s) changed.

────────────────────────────────────────────────────────────────
  [1/5]  row index 0
@@ -8,7 +8,5 @@
 </name>

 <name id="2" type="provider">

-<variations>



In [3]:
raw_df = pd.read_csv("../../data/test.csv")
raw_df

,anonymized,original,examples
0,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","(2024), Liam Novak (URL: https://myhealthonlin...",NONE
1,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","On 2023, Liam Novak, a 68y male patient with a...",NONE
2,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Dear Dr. Larsson, I hope this email finds you ...",NONE
3,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","Mr. Pt, email: d.kim1980@email.com, visited Me...",NONE
4,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, a 58year-old man residing at...",NONE
...,...,...,...
399,"<phi_entities>\n<names>\n<name id=""1"" type=""pr...","08-30-18, Pt, 68y man, presented to Cleveland ...",NONE
400,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","David Kim, a 28 years old male, visited the cl...",NONE
401,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","Mr. David Okafor, 28 years old, presented with...",NONE
402,"<phi_entities>\n<names>\n<name id=""1"" type=""pa...","David cKim, a 28 yo male, visited Mercy Hospit...",NONE


## 1 — Load Baseline Models

Comment out any baselines whose dependencies are not installed.

In [4]:
import datasets 

from baselines import PresidioBaseline, GLiNERBaseline, RobertaDeidBaseline, ScispaCyBaseline

models = [
    PresidioBaseline(score_threshold=0.5),
    #GLiNERBaseline(model_name="urchade/gliner_mediumv2.1", threshold=0.5),
    RobertaDeidBaseline(model_name="obi/deid_roberta_i2b2"),
    #ScispaCyBaseline(model_name="en_core_sci_lg"),
]
print("Loaded:", [m.name for m in models])

Device set to use mps:0


Loaded: ['Presidio', 'RoBERTa-i2b2']


In [3]:
import os

# Set proxy for standard HTTP/HTTPS traffic
os.environ["http_proxy"] = "http://127.0.0.1:10809"
os.environ["https_proxy"] = "http://127.0.0.1:10809"

# Set fallback proxy for all other traffic (requires httpx[socks])
os.environ["all_proxy"] = "socks5://127.0.0.1:10808"

print("Proxy environment variables set.")

Proxy environment variables set.


## 2 — Run Baselines

In [5]:
all_results = {}   # model_name -> EvalResult

for model in models:
    print(f"\nRunning {model.name} ...")
    predictions = [
        model.predict_xml(note)
        for note in tqdm(notes, desc=model.name)
    ]
    result = compute_metrics(
        predictions, gt_xmls, source_notes=notes,
        supported_entities=model.supported_entities,
    )
    all_results[model.name] = result
    print_report(result, title=model.name)


Running Presidio ...


Presidio: 100%|██████████| 377/377 [00:12<00:00, 30.74it/s]


 Presidio
  Samples evaluated : 377
  XML parse rate    : 1.0000

  Micro Precision   : 0.5344
  Micro Recall      : 0.6453
  Micro F1          : 0.5846
  F-β (β=2)         : 0.6196  (recall-weighted)
  Macro F1          : 0.6908

  Zero-leak rate    : 0.0451  (notes with no missed PHI)
  Hallucination rate: 0.4656  (1 − Precision)
  Token reduction   : 0.1413

Tag                 P        R       F1       GT     Pred
------------------------------------------------------------
address        0.4088   0.9953   0.5796      214      521
date           0.3618   0.7721   0.4927      588     1255
email          0.9953   0.9953   0.9953      215      215
name           0.6602   0.9785   0.7884     1114     1651
phone          0.9677   0.5430   0.6957      221      124
url            0.4220   0.9966   0.5929      293      692

Running RoBERTa-i2b2 ...


RoBERTa-i2b2:   0%|          | 0/377 [00:00<?, ?it/s]/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/transformers/pipelines/token_classification.py:490: UserWarning: Tokenizer does not support real words, using fallback heuristic
  warnings.warn(
RoBERTa-i2b2: 100%|██████████| 377/377 [00:30<00:00, 12.31it/s]

 RoBERTa-i2b2
  Samples evaluated : 377
  XML parse rate    : 1.0000

  Micro Precision   : 0.4978
  Micro Recall      : 0.7785
  Micro F1          : 0.6073
  F-β (β=2)         : 0.6996  (recall-weighted)
  Macro F1          : 0.6507

  Zero-leak rate    : 0.1883  (notes with no missed PHI)
  Hallucination rate: 0.5022  (1 − Precision)
  Token reduction   : 0.1459

Tag                 P        R       F1       GT     Pred
------------------------------------------------------------
address        0.2542   0.9907   0.4046      214      834
age            0.9763   0.9910   0.9836      332      337
date           0.5831   0.8350   0.6867      588      842
email          0.9913   0.5302   0.6909      215      115
id             0.5353   0.7845   0.6364      232      340
name           0.5277   0.9677   0.6829     1114     2043
phone          0.4876   0.9774   0.6506      221      443
place          0.3082   0.9883   0.4698      256      821


## 3 — Add Our Models

Load pre-computed predictions from the synth_eval / ext_eval notebooks
(run those notebooks first and save `*_predictions.csv`).

In [ ]:
import os

prefix = "synth_eval" if DATASET == "synth" else "ext_eval"

# SFT model predictions
sft_pred_path = f"../{prefix}/{prefix.replace('_eval','')}_eval_predictions.csv"
if os.path.exists(sft_pred_path):
    sft_df = pd.read_csv(sft_pred_path)
    sft_result = compute_metrics(
        sft_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes
    )
    all_results["EdgeDeID-SFT"] = sft_result
    print_report(sft_result, title="EdgeDeID-SFT")
else:
    print(f"SFT predictions not found at {sft_pred_path} — run synth_eval.ipynb first.")

# GRPO model predictions — update path after GRPO training
# grpo_pred_path = f"../{prefix}/grpo_predictions.csv"
# if os.path.exists(grpo_pred_path):
#     grpo_df = pd.read_csv(grpo_pred_path)
#     grpo_result = compute_metrics(grpo_df["LLM-XML"].tolist(), gt_xmls, source_notes=notes)
#     all_results["EdgeDeID-GRPO"] = grpo_result
#     print_report(grpo_result, title="EdgeDeID-GRPO")

SFT predictions not found at ../ext_eval/ext_eval_predictions.csv — run synth_eval.ipynb first.


## 4 — Comparison Table

In [6]:
rows = []
for model_name, r in all_results.items():
    rows.append({
        "Model":              model_name,
        "Micro-P":            round(r.micro_precision, 4),
        "Micro-R":            round(r.micro_recall, 4),
        "Micro-F1":           round(r.micro_f1, 4),
        "F-β(2)":             round(r.f_beta, 4),
        "Macro-F1":           round(r.macro_f1, 4),
        "Zero-leak":          round(r.zero_leak_rate, 4),
        "XML Parse Rate":     round(r.xml_parse_rate, 4),
        "Hallucination Rate": round(r.hallucination_rate, 4),
        "Token Reduction":    round(r.token_reduction_rate, 4) if r.token_reduction_rate == r.token_reduction_rate else None,
    })

comparison_df = pd.DataFrame(rows).set_index("Model")
comparison_df

,Micro-P,Micro-R,Micro-F1,F-β(2),Macro-F1,Zero-leak,XML Parse Rate,Hallucination Rate,Token Reduction
Model,,,,,,,,,
Presidio,0.5344,0.6453,0.5846,0.6196,0.6908,0.0451,1.0,0.4656,0.1413
RoBERTa-i2b2,0.4978,0.7785,0.6073,0.6996,0.6507,0.1883,1.0,0.5022,0.1459


In [7]:
import os
os.makedirs("results", exist_ok=True)

out_path = f"results/baseline_comparison_{DATASET}.csv"
comparison_df.to_csv(out_path)
print(f"Saved {out_path}")

Saved results/baseline_comparison_test.csv


## 5 — Per-Entity Breakdown

In [8]:
all_entity_tags = sorted(
    set(tag for r in all_results.values() for tag in r.per_entity)
)
per_entity_rows = []
for entity in all_entity_tags:
    row = {"Entity": entity}
    for model_name, r in all_results.items():
        m = r.per_entity.get(entity)
        row[model_name] = round(m.f1, 4) if m else None
    per_entity_rows.append(row)

per_entity_df = pd.DataFrame(per_entity_rows).set_index("Entity")

out_path = f"results/baseline_per_entity_{DATASET}.csv"
per_entity_df.to_csv(out_path)
print(f"Saved {out_path}")
per_entity_df

Saved results/baseline_per_entity_test.csv


,Presidio,RoBERTa-i2b2
Entity,,
address,0.5796,0.4046
age,NaN,0.9836
date,0.4927,0.6867
email,0.9953,0.6909
id,NaN,0.6364
name,0.7884,0.6829
phone,0.6957,0.6506
place,NaN,0.4698
url,0.5929,NaN
